# Downloading Uncalibrated Data

*Reminder:* When using ``BREADS`` for JWST, make sure you installed it with the JWST dependencies; see installation instruction (``pip install .[jwst]``).

In this tutorial we will use ```astropy.fits.open``` and/or ```jwst_mast_query``` to obtain ```uncal``` JWST data files which are needed for the BREADS analysis.

### Downloading public data (Easy)

If you know the exact filenames you would like to download and the datasets have completed their proprietary period to become public, it is straightforward to obtain them directly through the MAST API using ```astropy.fits.open```.

In [1]:
import astropy.io.fits as fits
import os

**This cell must be modified!** Specify the directory you would like to store data products in

In [6]:
base_path = os.path.join(os.getenv('BREADS_DATA'),"tutorial")

These subdirectories will be useful for organizing data products.

In [8]:
data_path = os.path.join(base_path,'GTO1414')
raw_path = os.path.join(data_path,'raw')
create_paths = [base_path,data_path,raw_path]
for path in create_paths:
    if not os.path.exists(path):
        os.mkdir(path)

The next cell only downloads 4 files (2 for nrs1 and 2 for nrs2) for testing purposes. See next

In [10]:
filelist = ['jw01414013001_02101_00001_nrs1_uncal.fits',
            'jw01414013001_02101_00001_nrs2_uncal.fits',
            'jw01414013001_02101_00002_nrs1_uncal.fits',
            'jw01414013001_02101_00002_nrs2_uncal.fits']
for file in filelist:
    print('downloading {} -> {}'.format(file,raw_path))
    mast_file_url = f"https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/{file}"
    hdul = fits.open(mast_file_url)
    hdul.writeto(os.path.join(raw_path,file),overwrite=True)

downloading jw01414013001_02101_00001_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414013001_02101_00001_nrs2_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414013001_02101_00002_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414013001_02101_00002_nrs2_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw


This loop iterates over a predefined sequence of observations from GTO program 1414, specifically sequences 12, 13, and 14, observations numbered 1-9, and both nrs1 and nrs2 detectors. The resulting files are downloaded and saved under the "raw" subdirectory.

In [11]:
for seq_num in ['012','013','014']:
    for obs_num in [str(x) for x in range(1,9+1)]:
        for det_string in ['nrs1','nrs2']:
            file = 'jw01414'+seq_num+'001_02101_0000'+obs_num+'_'+det_string+'_uncal.fits'
            print('downloading {} -> {}'.format(file,raw_path))
            mast_file_url = f"https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/{file}"
            hdul = fits.open(mast_file_url)
            hdul.writeto(os.path.join(raw_path,file),overwrite=True)

downloading jw01414012001_02101_00001_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00001_nrs2_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00002_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00002_nrs2_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00003_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00003_nrs2_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00004_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00004_nrs2_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414012001_02101_00005_nrs1_uncal.fits -> /stow/jruffio/data/breads_data/tutorial/GTO1414/raw
downloading jw01414

Thats it! If you just want to run the tutorial series, you can move onto notebook #2. If you are looking to download exclusive access data which is not public, the process is slightly more involved, but explained below.

### Downloading exclusive access data with jwst_mast_query (Medium)

In order to obtain uncalibrated data files for observations which are not yet public, it is required (to my knowledge) to first install ```jwst_mast_query``` (https://github.com/spacetelescope/jwst_mast_query). Additionally, one must have acess to a MAST account with the proper exclusive access dataset authorization (ask your PI for help.) Once this is in order, you can genereate a specific exclusive access token at (https://auth.mast.stsci.edu/token) to properly communicate your privileged status.

**This cell must be modified!** Replace the #### with your token.

In [25]:
token = "####"

os.environ["MAST_API_TOKEN"] = token

This cell will check your token is properly set as an environment varialbe.

In [23]:
import subprocess
return_token = subprocess.check_output('echo $MAST_API_TOKEN',shell=True)
assert token == str(return_token)[2:-3]

DL = 'y'The following cell uses the shell to call ```jwst_download.py``` with specific arguments for our purpose. You can modify the ```propID``` and ```date_string``` to select other sequences of observations. The current configuration will download the tutorial dataset from GTO 1414 on HD 19467 B.

**Replace DL = 'n' with 'y'** to proceed with the download. Otherwise it will simply print a list of files matching the search criteria.

In [24]:
DL = 'n' # change me to 'y' to download, 'n' to just print the list of files matching the search criteria
program = '1414' # change me
outsubdir = 'raw_from_jwst_download' # change me
date_string = '2024-01-01 2024-01-31' # change me

cmd = f"""\
while true; do echo {DL}; sleep 1; done | \
jwst_download.py \
--propID {program} \
--instrument nirspec \
--filetypes 'uncal' \
--outrootdir '{data_path}' \
--outsubdir '{outsubdir}' \
--skip_propID2outsubdir \
--date_select {date_string} \
"""
subprocess.call(cmd,shell=True)

INSTRUMENT:  nirspec
obsmode:  [None]
propID:  01414
obsnums:  None
INFO: MAST API token accepted, welcome Jean-Baptiste Ruffio [astroquery.mast.auth]
MJD range: 60310.0 60340.0
No obsmode given. Querying for all files for nirspec.
allowed filetype list: ['_uncal.fits']

######################
### Selected Products:
######################
 proposal_id obsnum     obsID  parent_obsid                         obs_id  sca visit dataproduct_type                           productFilename    filetype  calib_level      size                                   description
        1414     13 207380175     207409609 jw01414013001_02101_00001_nrs2 nrs2   001            image jw01414013001_02101_00001_nrs2_uncal.fits _uncal.fits            1 196657920 exposure (L1b): Uncalibrated 4D exposure data
        1414     12 207380194     207424082 jw01414012001_02101_00009_nrs1 nrs1   001            image jw01414012001_02101_00009_nrs1_uncal.fits _uncal.fits            1 196657920 exposure (L1b): Uncalibrate

0